# 16 - Figures: triage first, description second

> **Run order.** Step 16. Needs 04 (figures on disk) and 14 (the `ctx` index). Uses the local GPU
> for about an hour, and is **resumable**: re-running skips every figure already described.
> Logic lives in `src/analyst/vision.py`; see [ADR-010](../docs/adr/0010-figures.md).

The project name says *multimodal*, and until now the 418 extracted figures were stored and
never read. Two measurements came before any code:

- **The images are mostly not charts.** Four samples were a plant photo, blank line art, a CSR
  photo and a QR code.
- **Vector charts are invisible to image extraction**, and counting drawing paths does not find
  them either: the most path-heavy pages were leadership photo grids, icon-laden text pages and
  ruled statement tables.

So a local vision model first labels each image's **kind**, and only chart, table, infographic
and diagram are indexed. The description is generated text, so it is stored in its own table
with the model that wrote it - never in `elements.text`, which stays the filing's own words.

In [ ]:
from analyst.logging import configure_logging
configure_logging()

import pandas as pd
pd.set_option("display.width", 220)
pd.set_option("display.max_colwidth", 110)

from sqlalchemy import func, select

from analyst import evaluation as ev
from analyst.config import get_settings
from analyst.db import session_scope
from analyst.llm import LLM
from analyst.models import FigureDescription
from analyst.vision import KEEP, describe_all

settings = get_settings()
vision = LLM(settings, "ollama", "gemma3:4b")  # fits in 4 GB VRAM; qwen3-vl (5.7 GB) does not
questions = ev.load_questions(settings.data_dir / "benchmark" / "questions.jsonl")
print(vision.name)

## 1. Describe every figure

~8 seconds an image on the RTX 3050. One commit per image, so an interruption costs one image.

In [ ]:
import time

t0 = time.perf_counter()
new = describe_all(vision)
print(f"described {sum(new.values())} new figures in {time.perf_counter() - t0:.0f}s")

with session_scope() as s:
    kinds = dict(s.execute(select(FigureDescription.kind, func.count())
                           .group_by(FigureDescription.kind)).tuples().all())
table = pd.Series(kinds, name="figures").sort_values(ascending=False).to_frame()
table["indexed"] = [k in KEEP for k in table.index]
print(table.to_string())
print(f"\nindexed: {table[table.indexed].figures.sum()} of {table.figures.sum()}")

## 2. What the model said

Spot-check before trusting it: the kind decides what gets indexed, and every figure in a
description could end up verified as "printed in the evidence" by the agent.

In [ ]:
with session_scope() as s:
    rows = s.execute(select(FigureDescription.kind, FigureDescription.element_id,
                            FigureDescription.description)).all()
sample = (pd.DataFrame(rows, columns=["kind", "element", "description"])
          .groupby("kind").head(2).sort_values("kind"))
sample["element"] = sample["element"].str[-24:]
print(sample.to_string(index=False))

## 3. Index the informative figures

Added to the SAME `ctx` collection the agent searches. Point ids are UUIDv5 of the chunk id, so
the existing table and text points are untouched, and a figure chunk can be deleted by type.

In [ ]:
from analyst.indexing import load_chunks
from analyst.retrievers import open_store

figs = [c for c in load_chunks(with_context=True, with_figures=True) if c.type == "figure"]
embedder, store = open_store(settings, "bge-small", "ctx")
before = store.count()
if figs:
    store.upsert(figs, list(embedder.embed_documents([c.embed_text for c in figs])))
print(f"figure chunks: {len(figs)}   collection points {before:,} -> {store.count():,}")

## 4. Control: did the figures cost the benchmark anything?

Every benchmark answer is a table element. Adding ~100 new points could push some of them down
the ranking. Same questions, same retriever, same collection - now with figures in it.

In [ ]:
from analyst.retrievers import dense

search = dense(embedder, store, "ticker+year", expand=True)
cfg = ev.RunConfig(retriever="dense+expand[ctx+figures]", model="bge-small",
                   filters="ticker+year", limit=max(ev.K_VALUES), points=store.count(),
                   notes=f"+{len(figs)} figure chunks described by {vision.name}")
run = ev.build_run(cfg, ev.evaluate(questions, search, max(ev.K_VALUES)), questions,
                   deep=ev.evaluate(questions, search, max(ev.DEPTHS)), root=ev.ROOT)
ev.append_run(run)
ev.write_leaderboard(ev.load_runs())

ledger = [r for r in ev.load_runs()
          if r.config.retriever in ("dense+expand[ctx]", "dense+expand[ctx+figures]")]
print(pd.DataFrame([{**r.row(), **{f"@{d}": v for d, v in r.depth_curve.items()}}
                    for r in ledger]).drop(columns=["model", "filters", "bench"]).to_string())

## 5. Do figures come back for figure-shaped questions?

**Qualitative only - there is no ground truth for figures**, so this is a look, not a metric.

In [ ]:
for q, ticker in [("revenue trend chart over the years", "SUNPHARMA"),
                  ("infographic of key financial highlights", "HDFCBANK"),
                  ("business segments diagram", "RELIANCE")]:
    hits = store.search(embedder.embed_query(q), limit=10, ticker=ticker)
    top = [(i, h.pages[0], h.type) for i, h in enumerate(hits, 1) if h.type == "figure"]
    print(f"{ticker:<10} {q!r}: figure hits (rank, page) {top[:3]}")